# 1. Надійність абсолютної візуальної локалізації за синтетичного туману й диму

Мета — виміряти вплив видимості й неточності попереднього положення на **повний**
CAMP retrieval → RoMa matching → OpenCV P3P/RANSAC конвеєр.
Retrieval самостійно шукає ділянки карти; правильна ділянка не передається алгоритму.
PnP визначає положення камери за відповідностями між пікселями та 3D-точками карти.
RANSAC перевіряє геометричну узгодженість, відкидаючи помилкові відповідності.

Використовуються реальні зображення AnyVisLoc і аерофотознімальні карти; туман, дим та
попередні похибки синтетичні. Це не перевірка реальної INS/VIO або польоту в диму.
Навчання моделей, планування маршрутів та злиття вимірювань не входять у цей інкремент.

Перед запуском скопіюйте локальну папку `scripts/visibility_localization` у Google Drive.
Notebook імпортує модулі з цієї папки через `CODE_ROOT`; копій коду в комірках немає.
Середовище: Python 3.10–3.13, GPU з підтримкою CUDA 12.1 (Python ≤3.12) або 12.6 (Python 3.13), інтернет, місце для вибраних сцен.
Оберіть **Runtime → Change runtime type → GPU**. Якщо встановлення залежностей просить
перезапуск, перезапустіть сеанс і знову виконайте клітинки зверху вниз.

Стан під час створення: перевірки геометрії/оркестрації виконуються на синтетичних даних;
реальні CAMP/RoMa, завантаження даних та результати Colab потребують фактичного запуску.
[Джерело AnyVisLoc](https://github.com/UAV-AVL/Benchmark),
[ліцензія датасету](https://github.com/UAV-AVL/Benchmark/blob/main/docs/LICENSE_DATASET.md).
Не публікуйте похідні зображення без перевірки ліцензійних дозволів.

Поточний запуск: 24 кадри × 6 умов = 144 спроби, карта вирівнюється за yaw
попередньої оцінки орієнтації (профіль main, σyaw=5°). Точний yaw із GT
не передається в inference. Попередні результати збережено під окремим RUN_TAG.


## 2. Джерела, каталоги й параметри

**Наш код читається з папки в Google Drive, яку ви завантажуєте самостійно.**
`CODE_ROOT` — батьківський каталог пакета `visibility_localization`. Drive підключається для імпортів.
ZIP і розпаковані сцени залишаються на тимчасовому диску Colab.
`SAVE_RESULTS_TO_DRIVE=True` окремо вмикає постійне збереження результатів.

| Звідки | Куди | Що |
| --- | --- | --- |
| GitHub UAV-AVL/Benchmark | `WORK_ROOT/Benchmark` | Оригінальний код авторів |
| Офіційний OneDrive | `WORK_ROOT/archives` | ZIP тільки явно обраних сцен |
| ZIP | `WORK_ROOT/data` | Розпаковані файли сцен |
| Завантажена вами папка в Drive | `CODE_ROOT/visibility_localization` | Наші Python-модулі; notebook лише імпортує їх |
| Офіційне посилання CAMP у README | `Benchmark/Retrieval_Models/CAMP/weights` | Ваги CAMP |
| Офіційні URL RoMa Outdoor і DINOv2 із README | `Benchmark/Matching_Models/RoMa/ckpt` | Ваги matching |
| Експеримент | `RESULTS_ROOT/RUN_TAG` | CSV, JSON, графіки |

Читання CAMP із Google Drive авторів — це **не запис у ваш Drive**.
Усі адреси й каталоги виводяться перед початком роботи.

Діагностичні кадри зафіксовані в `DIAGNOSTIC_SAMPLES`: Scene_05/L05_0969,
Scene_08/L08_0049 та Scene_11/L11_0545. Відсутній кадр зупиняє перевірку.
`selected` містить 24 кадри: по шість із Scene_05/08/11/21. RUN_BATCH=True:
після контрольних 18 спроб виконується пакет із 144 спроб.

| Параметри CONFIG | Значення та підстава |
| --- | --- |
| seed | 42 — довільне фіксоване зерно для відтворюваності |
| nadir_max_deg / preferred_nadir_deg | 15° / 10° — вимога завдання |
| sample_count | 24 кадри: по шість із чотирьох сцен |
| sigma_xy_m / trials | 20 м на вісь; один повтор — збільшується саме кількість кадрів |
| profiles / active_profiles | Погоджені small/main/large; основний main |
| top_k / gallery_overlap / patch_scale / match_resize_ratio | 5 / 0,5 / 1,5 / 0,4 — однакові налаштування авторського конвеєра |
| retrieval_size / retrieval_batch | 384 px — вхід CAMP; 8 — пакет для пам'яті Colab |
| max_gallery_tiles / max_patch_pixels / max_image_side / gallery_cache_entries | 12000 / 4000000 / 1600 px / 2 — обмеження ресурсів, не наукові пороги |
| search_sigma | 3σ — початкова модель області пошуку |
| pnp_reprojection_px / pnp_iterations / pnp_confidence | 8 px / 2000 / 0,999 — параметри RANSAC |
| min_matches / min_inliers / min_inlier_ratio / max_median_reprojection_px / min_positive_depth_fraction | 12 / 8 / 0,25 / 4 px / 0,95 — початкове інженерне правило прийняття |
| success_threshold_m / recall_overlap_thresholds | 10 м / 25%, 50%, 75% — оцінювання після inference |
| fog / airlight_rgb / smoke | Параметри синтетичних умов; пояснення у розділі 6 |

`RUN_BATCH=True`: 24 кадри × 6 умов × одна σ × один повтор = **144 спроби**.
Перед пакетом виконуються окремі 18 контрольних спроб на попередніх трьох кадрах.
`SAVE_RESULTS_TO_DRIVE=True`: записи зберігаються у Drive для продовження запуску.
Окремі повторні inference для ілюстрацій вимкнено; графіки виміряних результатів залишаються.


In [ ]:
from pathlib import Path

# Фіксований робочий каталог цього notebook на тимчасовому диску Colab.
WORK_ROOT = Path('/content/visibility_localization')
UPSTREAM_ROOT = WORK_ROOT / 'Benchmark'  # код AnyVisLoc із GitHub
ARCHIVE_DIR = WORK_ROOT / 'archives'    # ZIP датасету
DATASET_ROOT = WORK_ROOT / 'data'       # розпаковані сцени
# Скопіюйте scripts/visibility_localization у цю папку Drive:
# /MyDrive/PhD/code/visibility_localization/
CODE_ROOT = Path('/content/drive/MyDrive/PhD/proj/scripts/')
PACKAGE_DIR = CODE_ROOT / 'visibility_localization'

# Drive потрібний для імпорту коду. Це окремий вибір місця для результатів.
SAVE_RESULTS_TO_DRIVE = True
RESULTS_ROOT = (Path('/content/drive/MyDrive/PhD/visibility_results')
                if SAVE_RESULTS_TO_DRIVE else WORK_ROOT / 'results')
RUN_TAG = 'visibility-v7-aerial-roma-yaw-24'
BASELINE_ROOT = RESULTS_ROOT / 'visibility-v5-aerial-roma-24'
RUN_BATCH = True
RUN_VISUAL_DIAGNOSTICS = False
DIAGNOSTIC_SAMPLES = [
    ('Scene_05', 'L05_0969'),
    ('Scene_08', 'L08_0049'),
    ('Scene_11', 'L11_0545'),
]

# Спочатку отримайте перелік у розділі 4. Попередні сцени не підставляються.
SCENES = ['Scene_05', 'Scene_08', 'Scene_11', 'Scene_21']
ARCHIVES = {
    'Scene_05': '/content/drive/MyDrive/PhD/proj/AnyVisLoc/Scene_05.zip',
    'Scene_08': '/content/drive/MyDrive/PhD/proj/AnyVisLoc/Scene_08.zip',
    'Scene_11': '/content/drive/MyDrive/PhD/proj/AnyVisLoc/Scene_11.zip',
    'Scene_21': '/content/drive/MyDrive/PhD/proj/AnyVisLoc/Scene_21.zip',
    }  # резерв: {'Scene_XX': '/content/drive/MyDrive/data/Scene_XX.zip'}
SCENE_URLS = {}  # резерв: перевірені прямі HTTPS-посилання на ZIP
CAMP_LOCAL_PATH = None  # резерв: шлях до офіційних ваг, які вже завантажено

# Офіційні джерела для довідки; їх використовує модуль colab.py нижче.
SOURCES = {
    'anyvisloc_code': 'https://github.com/UAV-AVL/Benchmark.git',
    'dataset_portal': 'https://1drv.ms/f/c/fef40436afe2e5bb/IgC0Zat90dsgTY-2lNlDliGXAfuTZOYEeKcllr7VPD_F3eY?e=LJ5fAD',
    'camp_weights': 'https://drive.google.com/file/d/1qHjXr3VVQuJZ5kE5u7YrUB8id90Nv2GJ/view',
    'roma_weights': 'https://github.com/Parskatt/storage/releases/download/roma/roma_outdoor.pth',
    'dinov2_weights': 'https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth',
}

CONFIG = {'seed': 42,
 'reference_mode': 'aerial',
 'matching_method': 'Roma',
 'map_alignment': 'prior_yaw',
 'scenes': [],
 'nadir_max_deg': 15.0,
 'preferred_nadir_deg': 10.0,
 'sample_count': 24,
 'pose_convention': 'ned_attitude',
 'sigma_xy_m': [20.0],
 'trials': 1,
 'profiles': {'small': {'yaw_deg': 2.0, 'tilt_deg': 1.0, 'z_m': 2.0},
              'main': {'yaw_deg': 5.0, 'tilt_deg': 2.0, 'z_m': 5.0},
              'large': {'yaw_deg': 15.0, 'tilt_deg': 5.0, 'z_m': 10.0}},
 'active_profiles': ['main'],
 'retrieval_modes': ['prior'],
 'top_k': 5,
 'search_sigma': 3.0,
 'gallery_overlap': 0.5,
 'patch_scale': 1.5,
 'retrieval_size': 384,
 'retrieval_batch': 8,
 'max_gallery_tiles': 12000,
 'max_patch_pixels': 4000000,
 'fallback_patch_sizes_m': [64.0, 128.0, 256.0],
 'max_image_side': 1600,
 'match_keypoints': 3000,
 'gallery_cache_entries': 2,
 'match_resize_ratio': 0.4,
 'pnp_reprojection_px': 8.0,
 'pnp_iterations': 2000,
 'pnp_confidence': 0.999,
 'min_matches': 12,
 'min_inliers': 8,
 'min_inlier_ratio': 0.25,
 'max_median_reprojection_px': 4.0,
 'min_positive_depth_fraction': 0.95,
 'success_threshold_m': 10.0,
 'recall_overlap_thresholds': [0.25, 0.5, 0.75],
 'fog': {'fog_weak': 0.8, 'fog_moderate': 0.55, 'fog_strong': 0.3},
 'airlight_rgb': [0.92, 0.94, 0.96],
 'smoke': {'smoke_weak': {'coverage': 0.35,
                          'transmission': 0.55,
                          'softness': 0.08,
                          'color_rgb': [0.65, 0.63, 0.6]},
           'smoke_strong': {'coverage': 0.7,
                            'transmission': 0.2,
                            'softness': 0.08,
                            'color_rgb': [0.5, 0.48, 0.45]}}}
CONFIG['scenes'] = SCENES
print('ДЖЕРЕЛА:')
for name,url in SOURCES.items(): print(name,'->',url)
print('КАТАЛОГИ COLAB:')
for name,path in [('Код AnyVisLoc',UPSTREAM_ROOT),('ZIP датасету',ARCHIVE_DIR),
                  ('Розпаковані дані',DATASET_ROOT),('Наш код у Drive',PACKAGE_DIR),('Результати',RESULTS_ROOT)]:
    print(name,'->',path)
print('Результати зберігатимуться у Drive:',SAVE_RESULTS_TO_DRIVE)

print('ПЕРЕВІРКА:', CONFIG['reference_mode'], '+ CAMP +', CONFIG['matching_method'], '+ P3P/RANSAC')
print('КАДРИ:', DIAGNOSTIC_SAMPLES)
print('RUN_TAG:', RUN_TAG, '; RUN_BATCH:', RUN_BATCH)

print('ПАКЕТ: 24 кадри, по 6 на сцену, 6 умов = 144 спроби; перед ним контрольні 18 спроб.')

print('Вирівнювання карти:', CONFIG['map_alignment'], '; yaw із noisy prior, профіль main: σyaw=5°')
print('Попередній запуск для порівняння:', BASELINE_ROOT)


## 3. Підключення папки коду та середовища

1. Скопіюйте локальну папку `scripts/visibility_localization/` цілком у `MyDrive/PhD/proj/scripts/`.
2. Якщо вибрали інше місце, змініть `CODE_ROOT` у розділі 2.
3. Комірка нижче підключить Drive, перевірить файли та додасть CODE_ROOT до шляху імпорту.
4. Залежності встановлюються за `requirements-colab.txt` із цієї папки.

Notebook не створює Python-модулі. Кеші спрямовані на диск Colab.
Після оновлення модулів у Drive перезапустіть Python-сеанс.

Код AnyVisLoc клонується окремо й не редагується. Він імпортується з каталогу checkout
через шлях пошуку Python, без встановлення самого репозиторію через pip.

Python 3.13: PyTorch 2.6.0 / torchvision 0.21.0 (cu126), NumPy 2.1.3.
Python 3.10–3.12: PyTorch 2.5.1 / torchvision 0.20.1 (cu121), NumPy 1.26.4.
Після оновлення notebook скопіюйте також новий `requirements-colab.txt` у Drive.


In [ ]:
import os, sys, json, subprocess
from google.colab import drive

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
print(sys.version_info[:2])
drive.mount('/content/drive')

required_files = (
    'alignment.py', 'test_alignment.py', '__init__.py', 'core.py', 'data.py', 'backend.py', 'campaign.py',
    'colab.py', 'analysis.py', 'check_upstream.py', 'test_experiment.py',
    'requirements-colab.txt', 'requirements-roma-colab.txt', 'models.py', 'test_matcher_selection.py', 'test_batch_sampling.py',
)
missing = [name for name in required_files if not (PACKAGE_DIR/name).is_file()]
if missing:
    raise FileNotFoundError(
        f'У {PACKAGE_DIR} відсутні файли: {missing}. '
        'Скопіюйте всю локальну папку scripts/visibility_localization або виправте CODE_ROOT.'
    )

# Байткод і кеші не записуються до завантаженої папки в Drive.
sys.pycache_prefix = str(WORK_ROOT/'pycache')
os.environ['PYTHONPYCACHEPREFIX'] = sys.pycache_prefix
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

import visibility_localization
loaded_package = Path(visibility_localization.__file__).resolve().parent
if loaded_package != PACKAGE_DIR.resolve():
    raise RuntimeError(f'Імпортовано іншу копію: {loaded_package}. Перезапустіть Python-сеанс.')
print('Модулі імпортовано з:', loaded_package)


In [ ]:
import sys, subprocess, importlib.metadata
from pathlib import Path
from packaging.requirements import Requirement

python_version = sys.version_info[:2]
if not (3, 10) <= python_version <= (3, 13):
    raise RuntimeError(f'Підтримується Python 3.10–3.13; поточний: {sys.version}')
if python_version == (3, 13):
    torch_version, vision_version, cuda_build = '2.6.0', '0.21.0', 'cu126'
    numpy_version = '2.1.3'
else:
    torch_version, vision_version, cuda_build = '2.5.1', '0.20.1', 'cu121'
    numpy_version = '1.26.4'
print(f'Python {sys.version.split()[0]}: torch {torch_version}, '
      f'torchvision {vision_version}, {cuda_build}, NumPy {numpy_version}', flush=True)
requirements = PACKAGE_DIR / 'requirements-colab.txt'
# Перевіряємо також копію requirements у Drive до будь-яких змін середовища.
active_requirements = [Requirement(line) for line in requirements.read_text().splitlines()
                       if line.strip() and not line.lstrip().startswith('#')]
if not any(r.name == 'numpy' and (r.marker is None or r.marker.evaluate())
           and r.specifier.contains(numpy_version) for r in active_requirements):
    raise RuntimeError('Оновіть requirements-colab.txt у Drive з локальної папки '
                       'scripts/visibility_localization та повторіть комірку.')

# Зміна бібліотеки, вже імпортованої в ядро, потребує перезапуску.
modules = {'numpy': 'numpy', 'scipy': 'scipy', 'pandas': 'pandas',
           'matplotlib': 'matplotlib', 'Pillow': 'PIL', 'shapely': 'shapely',
           'scikit-image': 'skimage', 'torch': 'torch', 'torchvision': 'torchvision',
           'opencv-python-headless': 'cv2', 'timm': 'timm', 'kornia': 'kornia'}
def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
loaded_before = {name: installed_version(name) for name, module in modules.items()
                 if module in sys.modules}
pins = [f'torch=={torch_version}', f'torchvision=={vision_version}']
WORK_ROOT.mkdir(parents=True, exist_ok=True)
constraints = WORK_ROOT / 'torch-constraints.txt'
constraints.write_text('\n'.join(pins) + '\n')
# Без -q: справжня причина помилки pip залишиться у виводі комірки.
subprocess.run([sys.executable, '-m', 'pip', 'install', *pins,
                '--index-url', f'https://download.pytorch.org/whl/{cuda_build}'], check=True)
install_requirements = (PACKAGE_DIR/'requirements-roma-colab.txt'
                        if CONFIG.get('matching_method', 'SP_LG') == 'Roma' else requirements)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(install_requirements),
                '-c', str(constraints)], check=True)
changed = [name for name, version in loaded_before.items()
           if installed_version(name) != version]
restart_message = ('Runtime → Restart session, потім виконайте notebook зверху вниз.')
if changed:
    raise RuntimeError(f'Оновлено вже імпортовані бібліотеки: {changed}. ' + restart_message)
import numpy as np, torch, torchvision, cv2
if (np.__version__ != numpy_version
        or torch.__version__.split('+')[0] != torch_version
        or torchvision.__version__.split('+')[0] != vision_version
        or cv2.__version__ != '4.10.0'):
    raise RuntimeError('Версії в ядрі не відповідають встановленим. ' + restart_message)
assert torch.cuda.is_available(), 'GPU CUDA недоступний; перевірте тип runtime'
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)
# Зберігаємо stdout/stderr pytest: деякі клієнти не показують вивід дочірнього процесу.
test_command = [sys.executable, '-m', 'pytest', '-q', '--tb=short',
                str(PACKAGE_DIR/'test_experiment.py'), str(PACKAGE_DIR/'test_matcher_selection.py'),
                str(PACKAGE_DIR/'test_batch_sampling.py'), str(PACKAGE_DIR/'test_alignment.py'),
                '-o', 'cache_dir=/content/visibility_pytest_cache']
test_process = subprocess.run(test_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', check=False)
test_log = WORK_ROOT/'setup-pytest.log'
test_log.write_text(test_process.stdout, encoding='utf-8')
print(test_process.stdout, flush=True)
print('Звіт тестів:', test_log, flush=True)
if SAVE_RESULTS_TO_DRIVE:
    drive_test_log = Path(RESULTS_ROOT)/'setup-pytest.log'
    drive_test_log.parent.mkdir(parents=True, exist_ok=True)
    drive_test_log.write_text(test_process.stdout, encoding='utf-8')
    print('Копія звіту в Drive:', drive_test_log, flush=True)
if test_process.returncode:
    raise RuntimeError(f'Автоматичні тести не пройшли (код {test_process.returncode}). '
                       f'Причина — у звіті вище та у {test_log}. Наступні розділи не запускати.')
from visibility_localization.colab import checkout_upstream
upstream = checkout_upstream(UPSTREAM_ROOT)
from visibility_localization.check_upstream import verify
print('Синтетична перевірка авторських інтерфейсів:', verify(upstream))
output_root = Path(RESULTS_ROOT)/RUN_TAG
output_root.mkdir(parents=True, exist_ok=True)


## 4. Завантаження та перевірка датасету

Спочатку пробуємо офіційний OneDrive: читаємо метадані й завантажуємо тільки ZIP
вибраних сцен за адресами, які реально повернув сервіс. Адреси архівів не вгадуються.
HTTP 401/403, змінена структура або неоднозначні назви зупиняють автоматичний шлях
і записуються в `acquisition.json`. За потреби задайте `ARCHIVES` у розділі 2
або `DATASET_ROOT` із уже розпакованими сценами та повторіть цю клітинку.
Посилання OneDrive тимчасові, автоматичний доступ без входу не гарантований.

[Офіційний портал](https://1drv.ms/f/c/fef40436afe2e5bb/IgC0Zat90dsgTY-2lNlDliGXAfuTZOYEeKcllr7VPD_F3eY?e=LJ5fAD).
Перевіряються CRC ZIP, безпечні шляхи та вільне місце. Ідентичні файли використовуються
повторно; змінені не перезаписуються. Контрольні суми джерел потрапляють у manifest.


In [ ]:
from visibility_localization.colab import official_scene_links,acquire_dataset
from visibility_localization.data import select_samples,read_sample,load_reference
existing=sorted({p.parent.name for p in Path(DATASET_ROOT).rglob('L??_reference.json')})
print('Уже розпаковані сцени:',existing)
if not CONFIG['scenes']:
    try:
        available=official_scene_links(None)
        print('ZIP в офіційному порталі:')
        for scene,item in sorted(available.items()): print(scene,item['name'],item.get('size'),'bytes')
    except Exception as exc:
        print('Інвентар недоступний:',type(exc).__name__,str(exc))
        print('Офіційний портал:',SOURCES['dataset_portal'])
        print('Резерв: ARCHIVES або вже розпакований DATASET_ROOT.')
    raise ValueError('Оберіть SCENES у розділі 2. Жоден ZIP автоматично не завантажено.')
print('Завантажуємо лише:',CONFIG['scenes'])
print('ZIP ->',ARCHIVE_DIR,'; розпакування ->',DATASET_ROOT)
acquisition=acquire_dataset(CONFIG['scenes'],ARCHIVE_DIR,DATASET_ROOT,
    output_root/'acquisition.json',archives=ARCHIVES,urls=SCENE_URLS)
print(json.dumps(acquisition,ensure_ascii=False,indent=2))


## 5. Відбір надирних і білянадирних зображень

Надир — напрям вертикально вниз. Кут визначається за оптичною віссю `R[:,2]`
у системі East/North/Up: `acos(R[:,2] · [0,0,-1])`.
Максимум 15°, перевага ≤10°. Euler pitch не використовується як кут від надиру.

За замовчуванням перевіряється відоме представлення випуску: матриця ZYX/NED
узгоджується з `euler_deg`, після чого враховується монтаж камери.
NED означає North/East/Down; внутрішня система конвеєра — East/North/Up.
Якщо випуск містить іншу конвенцію, перевірка блокує кадр: перегляньте
`selection_audit.csv`; не перемикайте представлення навмання.

Вибірка збалансована між сценами та фіксується до запуску локалізації.
Перевіряються всі NPZ вибраних сцен; за потреби це займає час.
Помилки читання й геометрії лишаються в аудиті. Результат локалізації на відбір не впливає.


In [ ]:
import pandas as pd
from visibility_localization.data import select_diagnostic_samples, validate_balanced_selection
import matplotlib.pyplot as plt
selected, selection_audit = select_samples(DATASET_ROOT, CONFIG, output_root/'selection')
# Перевіряємо збіг кадрів із попереднім запуском без повороту карти.
baseline_selection_path = BASELINE_ROOT/'selection/selected_samples.csv'
if not baseline_selection_path.is_file():
    raise FileNotFoundError(f'Потрібна попередня вибірка: {baseline_selection_path}')
baseline_selection = pd.read_csv(baseline_selection_path)
frame_keys = ['scene_id', 'sample_id']
if set(map(tuple, baseline_selection[frame_keys].to_numpy())) != {
        (r['scene_id'], r['sample_id']) for r in selected}:
    raise RuntimeError('Кадри відрізняються від попереднього запуску; порівняння зупинено')
display(pd.DataFrame(selected)[frame_keys])
coverage = validate_balanced_selection(selected, SCENES, CONFIG['sample_count'])
display(coverage)
coverage.to_csv(output_root/'selection/scene_coverage.csv', index=False)

display(selection_audit.groupby('scene_id').agg(frames=('sample_id','size'), eligible=('eligible','sum')))
display(selection_audit.loc[~selection_audit.eligible, ['scene_id','sample_id','error']].head(10))
selection_audit['nadir_deg'].dropna().hist(bins=30)
plt.xlabel('Кут від надиру, градуси'); plt.ylabel('Кількість кадрів'); plt.show()
diagnostic_selection = select_diagnostic_samples(selected, DIAGNOSTIC_SAMPLES)
print('Зафіксовані діагностичні кадри:', [(r['scene_id'], r['sample_id']) for r in diagnostic_selection])
example = read_sample(diagnostic_selection[0]['path'], CONFIG['pose_convention'], CONFIG['max_image_side'])
plt.imshow(example['rgb']); plt.title(f"{example['sample_id']}: {example['nadir_deg']:.2f}°"); plt.axis('off'); plt.show()
print('Відібрано:', len(selected), 'кадрів; перелік збережено до запуску локалізації')


## 6. Синтетичний туман і дим

Модель: **I(x) = J(x)t(x) + A(1−t(x))**. J — початкове зображення,
t — пропускання, A — атмосферна складова освітлення; інтенсивності нормовані до [0,1].
Туман однорідний: t=0,8/0,55/0,3. Дим: плавне випадкове поле з охопленням
35%/70% та мінімальним пропусканням близько 0,55/0,2. Колір і плавність задано в CONFIG.
Поле 64×64 згладжується та масштабується; однакова базова випадкова структура
використовується для обох інтенсивностей. Поріг охоплення задається квантилем поля.

Це змішування у просторі значень sRGB, без радіометричного калібрування та глибини.
Воно не визначає реальну метеорологічну видимість і не відтворює фізичне поширення диму.
Змінюється лише RGB зображення БПЛА; карта, калібрування, геометрія й еталон незмінні.


In [ ]:
from visibility_localization.analysis import show_visibility
visibility_fig, transmission_fig = show_visibility(example, CONFIG)
plt.show()
(output_root/'figures').mkdir(exist_ok=True)
visibility_fig.savefig(output_root/'figures/visibility_examples.png', dpi=140)
transmission_fig.savefig(output_root/'figures/transmission_examples.png', dpi=140)


## 7. Генерація попередніх оцінок положення та орієнтації

`x_prior=x_gt+εx`, `y_prior=y_gt+εy`, де εx,εy ~ N(0,σXY²), незалежні.
Висотна координата також отримує незалежну адитивну нормальну похибку.
Орієнтація: `R_prior = R_yaw · Exp([δtilt_E,δtilt_N,0]) · R_gt`.
Exp перетворює вектор малого повороту на матрицю; це уникає виродження Euler біля надиру.
Похибки нульового середнього не моделюють постійне зміщення, кореляцію або дрейф у часі.

Еталон використовується тут лише для моделювання вимірювань. Після цього функція
локалізації отримує тільки `Prior`: координати, матрицю обертання й задані σ.
Положення, поворот і seed однакові в усіх шести умовах одного парного випробування.
Непридатну геометрію зашумлених вимірювань відхиляємо, а не виправляємо еталоном.


In [ ]:
from visibility_localization.core import make_prior
prior, prior_metadata = make_prior(example['xyz_enu'], example['rotation'], 20.,
                                  CONFIG['profiles']['main'], CONFIG['seed'])
print(json.dumps(prior_metadata, ensure_ascii=False, indent=2))


## 8. Параметри експерименту та геометрія області пошуку

Приблизна проєкція центрального квадратного обрізання на площину з локальною висотою
DSM біля попереднього положення (медіана сцени — лише резерв поза DSM) задає центр спостереження та розмір фрагментів. Це наближення площиною;
воно не описує окремі будинки та паралакс. DSM не замінюється площиною у PnP:
для кожної зіставленої точки береться її доступна висота DSM.

У режимі `prior` залишаються ділянки, які перетинають круг радіуса `3σXY` навколо
**оціненого** центра спостереження. Круг моделює лише горизонтальну невизначеність;
додаткова невизначеність центра від кутів/висоти не входить у цей радіус.
Круг 3σ містить приблизно 98,89% маси двовимірного ізотропного нормального розподілу.
`global` шукає на всій доступній карті сцени. Вибір карти сцени вважається заданим;
це не пошук серед невідомих територій усього світу.

Якщо проєкція неможлива, пошук переходить до ділянок 64/128/256 м навколо
попереднього XY-положення. Це початкові інженерні масштаби, не виміряна висота.
Завеликі растрові ділянки обробляються зі зменшеною роздільністю для retrieval;
matching/PnP використовує координати оригінальної карти й DSM.
`geometry_fallback`, `retrieval_plane_source` і `retrieval_scales` зберігають цей шлях.

Перед CAMP і RoMa карта повертається авторською `dumpRotateImage` за yaw
із `prior.camera_to_enu`. Галерея будується на повернутій карті; географічний
фільтр переводить її ділянки назад у вихідну систему. Після matching авторська
зворотна матриця повороту повертає відповідності в початкові пікселі карти.
DSM залишається у своїй вихідній системі координат.


In [ ]:
import copy
from visibility_localization.core import footprint
from visibility_localization.backend import retrieval_geometry
reference = load_reference(diagnostic_selection[0]['reference_json'], CONFIG['reference_mode'])
center, scales, geometry = retrieval_geometry(reference, prior, example['K'], example['dist'],
                                              example['rgb'].shape, CONFIG)
print('Геометрія пошуку:', geometry)
fig, ax = plt.subplots(figsize=(6,6))
if not geometry['geometry_fallback']:
    region, _ = footprint(prior.xyz, prior.camera_to_enu, example['K'], example['dist'],
                          example['rgb'].shape, geometry['retrieval_plane_z_m'], center_crop=True)
    x,y = region.exterior.xy
    ax.plot(x,y,label='Проєкція з попередніх вимірювань')
ax.add_patch(plt.Circle(center, CONFIG['search_sigma']*prior.sigma_xy_m, fill=False, label='Круг пошуку'))
ax.scatter(*prior.xyz[:2],label='Попереднє положення')
ax.set_aspect('equal'); ax.autoscale_view(); ax.legend()
ax.set(xlabel='East, м',ylabel='North, м'); plt.show()


## 9. Завантаження моделей CAMP та RoMa

CAMP checkpoint завантажується за офіційним посиланням із README через gdown;
за обмеження сервісу задайте CAMP_LOCAL_PATH у розділі 2. Backend завантажує
RoMa Outdoor і DINOv2 ViT-L/14 за офіційними URL авторів, якщо їх ще немає
в `Benchmark/Matching_Models/RoMa/ckpt`, і викликає авторську `matching_init`.
CAMP може також завантажити початковий backbone через timm.
Моделі створюються один раз; навчання відсутнє. Зберігаються хеші checkpoint
і фактичних станів CAMP/RoMa. Для вибраного RoMa SP/LightGlue не ініціалізується.
Помилка ваг/імпорту/GPU зупиняє підготовку; matcher автоматично не підміняється.

Завантаження RoMa/DINOv2 повторюється до чотирьох разів після тимчасових
мережевих помилок (зокрема HTTP 504). Успішно отримані ваги кешуються
в `MyDrive/PhD/model_cache/anyvisloc`; новий сеанс копіює їх із кешу.
Якщо сервер недоступний після повторів, помилка показує конкретний
файл, офіційний URL і шлях для ручного розміщення оригінальних ваг.


In [ ]:
# Постійний кеш RoMa/DINOv2 переживає завершення сеансу Colab.
import os
MODEL_CACHE = Path('/content/drive/MyDrive/PhD/model_cache/anyvisloc')
os.environ['VISIBILITY_MODEL_CACHE'] = str(MODEL_CACHE)
print('Постійний кеш ваг:', MODEL_CACHE)
from visibility_localization.colab import CAMP_URL
checkpoint = Path(CAMP_LOCAL_PATH) if CAMP_LOCAL_PATH else Path(UPSTREAM_ROOT)/'Retrieval_Models/CAMP/weights/weights_0.9446_for_U1652.pth'
if not checkpoint.is_file():
    import gdown
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    downloaded = gdown.download(CAMP_URL, str(checkpoint), fuzzy=True, quiet=False)
    if not downloaded:
        raise RuntimeError('CAMP не завантажено. Задайте CAMP_LOCAL_PATH із Google Drive.')
from visibility_localization.backend import AnyVisLocBackend
backend = AnyVisLocBackend(upstream, CONFIG, checkpoint)
print(json.dumps(backend.identity, ensure_ascii=False, indent=2))

print('Фактичний matcher:', backend.identity['matching_method'], '; карта/DSM:', CONFIG['reference_mode'])
assert backend.identity['matching_method'] == CONFIG['matching_method']


## 10. Авторський повний конвеєр із run_avl.py

Клонований каталог AnyVisLoc додається до `sys.path` — списку каталогів, де Python
шукає модулі. `importlib.import_module('avl_utils')` завантажує авторський модуль.
Файли AnyVisLoc не змінюються, пакувальні метадані не додаються.

Використовуються `retrieval_init`, `matching_init`, `retrieval_all_anyvisloc`,
`Match2Pos_all_anyvisloc` та `estimate_drone_pose_anyvisloc` з AnyVisLoc.

`bind_upstream` створює окрему функцію з тим самим скомпільованим тілом і копією
словника глобальних імен. У цій копії замінюються лише залежності для фільтра сітки,
геометрії prior, кешу та збирання показників PnP. Оригінальна функція й модуль не змінюються.
Це адаптація поведінки на час виконання, а не просто інший синтаксис імпорту.

Matching отримує `truePos=None`. Еталон не використовується для ранжування або прийняття.
Режим правильного crop `run_avl_match_loc.py` не використовується. Карта повертається за шумним yaw із prior; matching використовує режим yp
для зворотного перетворення координат. Геометрія пошуку залишається нашою адаптацією.

Правило прийняття: ≥12 відповідностей, ≥8 inliers, частка ≥0,25,
медіанна репроєкція ≤4 px, ≥95% точок перед камерою. Пороги фіксовані для всіх умов.
Авторський extractor обчислює галерею та запит разом, окремі часи не вигадуються.


In [ ]:
import inspect
from visibility_localization.backend import localize
print(inspect.signature(localize))


## 11. Перевірка відсутності еталонних підказок

Інтерфейс inference не отримує sample/GT, еталонні Euler або координати.
У розділі 3 виконано синтетичний тест: незалежна зміна еталона в оцінювачі
змінює лише оцінену правильність, а не вихід локалізації.
Нижче додаткова перевірка на справжніх зображеннях і моделях після їх завантаження.
Карта захищена від запису; контрольна сума масиву перевіряється до/після.


In [ ]:
from visibility_localization.core import score_result, array_hash
map_before = array_hash(reference.bgr)
a = localize(backend, example['rgb'], example['K'], example['dist'], reference, prior, 'prior', 314)
good_evaluation = score_result(a, example['xyz_enu'])
changed_evaluation = score_result(a, example['xyz_enu']+np.array([10000,10000,0]))
b = localize(backend, example['rgb'], example['K'], example['dist'], reference, prior, 'prior', 314)
assert a['accepted'] == b['accepted'] and a['candidates'] == b['candidates']
if a['pred_xyz_enu'] is not None:
    np.testing.assert_allclose(a['pred_xyz_enu'], b['pred_xyz_enu'], rtol=0, atol=1e-6)
assert array_hash(reference.bgr) == map_before
print('Перевірку виконано на поточних моделях; еталон не є входом inference.')


## 12. Запуск сценаріїв

Кожна спроба має окремий JSON із контрольною сумою. Перерваний запуск продовжується
без повторення завершених спроб, включно з відмовами. Зміна коду/даних/параметрів/
бібліотек вимагає нового RUN_TAG. Технічні помилки мають окремий статус;
`retry_technical=True` дозволяє повторити їх зі збереженням історії.

Діагностика на 2–3 кадрах перевіряє виконання і не підтримує наукових узагальнень.
Основна частка успіхів включає алгоритмічні відмови у знаменник; технічні помилки
показуються окремо з кількістю спроб. Без валідної пози похибка порожня (NaN у CSV).

Перед кампанією звільняються демонстраційні карти й фігури. Стару повернуту
карту видалено до створення наступної; хешування не створює повнорозмірних
байтових копій. `resource_log.jsonl` записує RAM/GPU та останню спробу
перед inference і після збереження результату. Раптове завершення ядра
не завжди залишає Python traceback; цей журнал допомагає локалізувати збій.

Після зміни коду використовується новий RUN_TAG v7. Записи v6 залишаються
недоторканими; змішування результатів різних версій коду блокується.


In [ ]:
# Звільняємо великі карти/фігури попередніх демонстрацій перед кампанією.
import gc
plt.close('all')
reference = None
for temporary_name in ['a','b','visibility_fig','transmission_fig']:
    globals().pop(temporary_name, None)
backend.release_reference()
gc.collect()
from visibility_localization.campaign import run_experiment
from visibility_localization.data import select_diagnostic_samples
from visibility_localization.core import CONDITIONS
import pandas as pd

diagnostic_config = copy.deepcopy(CONFIG)
diagnostic_config.update(sample_count=3, sigma_xy_m=[20.], trials=1, active_profiles=['main'])
diagnostic_selection = select_diagnostic_samples(selected, DIAGNOSTIC_SAMPLES)
expected_attempts = (len(diagnostic_selection)*len(CONDITIONS)*len(diagnostic_config['sigma_xy_m'])*
                     diagnostic_config['trials']*len(diagnostic_config['active_profiles'])*
                     len(diagnostic_config['retrieval_modes']))
if expected_attempts != 18:
    raise ValueError(f'Ця діагностика має містити рівно 18 спроб; налаштовано {expected_attempts}')
print('Карта/DSM:', diagnostic_config['reference_mode'], '; matcher:', diagnostic_config['matching_method'])
print('Кадри:', [(r['scene_id'], r['sample_id']) for r in diagnostic_selection])
print('Очікується спроб:', expected_attempts, '; вирівнювання карти:', CONFIG['map_alignment'], '; σyaw=5°')
diagnostic_results = run_experiment(backend, diagnostic_selection, diagnostic_config,
                                    output_root/'diagnostic')
if len(diagnostic_results) != expected_attempts:
    raise RuntimeError('Неповний діагностичний запуск: перевірте progress.json')
view = diagnostic_results.copy()
view['condition'] = pd.Categorical(view.condition, categories=CONDITIONS, ordered=True)
display(view.sort_values(['condition','scene_id'])[[
    'scene_id','sample_id','reference_mode','matching_method','condition','status','accepted',
    'error_xy_m','raw_matches','matches','inliers','rejection_reason','geometry_fallback']])
status_counts = diagnostic_results.status.value_counts().reindex(
    ['correct','false_accept','rejected','technical_error'], fill_value=0).rename_axis('status').to_frame('attempts')
display(status_counts)
status_counts.to_csv(output_root/'diagnostic/status_counts.csv')
if diagnostic_results.status.eq('technical_error').any():
    raise RuntimeError('Є технічні помилки: перегляньте records/*.json перед аналізом')


In [ ]:
if RUN_BATCH:
    from visibility_localization.data import validate_balanced_selection
    from visibility_localization.core import CONDITIONS
    from visibility_localization.campaign import read_json
    coverage = validate_balanced_selection(selected, SCENES, CONFIG['sample_count'])
    expected_batch_attempts = (len(selected)*len(CONDITIONS)*len(CONFIG['sigma_xy_m'])*CONFIG['trials']*
                               len(CONFIG['active_profiles'])*len(CONFIG['retrieval_modes']))
    if expected_batch_attempts != 144:
        raise ValueError(f'Погоджений пакет: 24 кадри × 6 умов = 144 спроби; налаштовано {expected_batch_attempts}')
    print('Карта/DSM:', CONFIG['reference_mode'], '; matcher:', CONFIG['matching_method'],
          '; вирівнювання:', CONFIG['map_alignment'], '; yaw: noisy prior, σ=5°')
    print('ПАКЕТ:', expected_batch_attempts, 'спроб; кадрів:', len(selected))
    display(coverage)
    elapsed = pd.to_numeric(diagnostic_results.get('total_s', pd.Series(dtype=float)), errors='coerce').dropna()
    if len(elapsed):
        print('Орієнтовний час inference за поточною діагностикою, хв:',
              round(float(elapsed.median())*expected_batch_attempts/60, 1))
    print('Постійний каталог результатів:', output_root/'batch')
    print('Після переривання повторіть із тими самими параметрами й RUN_TAG; завершені записи використовуються повторно.')
    results = run_experiment(backend, selected, CONFIG, output_root/'batch')
    measured_output = output_root/'batch'
    progress = read_json(measured_output/'progress.json')
    if len(results) != expected_batch_attempts or not progress['complete']:
        raise RuntimeError('Пакет не завершено: перевірте progress.json і повторіть запуск без зміни конфігурації')
else:
    results = diagnostic_results
    measured_output = output_root/'diagnostic'
    print('Пакетний запуск вимкнений; нижче лише технічна діагностика.')


## 13. Експорт результатів

`per_sample_results.csv` містить усі спроби, вкладені поля серіалізовані як JSON.
`experiment_config.json` зберігає конфігурацію, commit, ваги, версії та джерела;
`records/` — первинні результати, `progress.json` — завершеність.
Еталон, фактичні похибки вимірювань і оцінювання retrieval зберігаються після inference.
Дані синтетичних умов відновлюються з CONFIG і seed; усі зображення не записуються.
Якщо RESULTS_ROOT розташований у Drive, збереження вже є постійним.


In [ ]:
from visibility_localization.campaign import export_records
results = export_records(measured_output)
print('Результати:', measured_output)
print('Спроб:', len(results), 'Технічних помилок:', results.status.eq('technical_error').sum())

status_counts = results.status.value_counts().reindex(
    ['correct','false_accept','rejected','technical_error'], fill_value=0).rename_axis('status').to_frame('attempts')
display(status_counts)
status_counts.to_csv(measured_output/'status_counts.csv')
print('Унікальних кадрів:', len(results[['scene_id','sample_id']].drop_duplicates()))


## 14. Статистичний аналіз та графіки

Коректна локалізація: прийнята позиція з XY-похибкою ≤10 м.
Хибне прийняття: прийнята позиція з похибкою >10 м. Відмова: позиція не прийнята.
Додатково Acc@3/5/10/20 м. Середня/медіанна/95-й процентиль похибки —
**лише серед прийнятих позицій**, включно з хибними; кількість спостережень явна.

Recall@K перевіряє наявність серед K кандидатів ділянки з перекриттям видимої поверхні
не менше 25/50/75%; основний поріг 50%. Це не гарантія успішного matching/PnP.
Перекриття використовує центральний квадрат, який реально отримав CAMP, і наближення
площиною; потрапляння центра зберігається додатково. Еталон використовується лише тут.
`paired_differences.csv` порівнює кожну умову з clear за тим самим prior.
Технічно невалідні пари виключені з різниць; різниця похибок є тільки коли обидва рішення прийняті.
Сусідні кадри й повтори одного кадру не є незалежними територіями: графіки описові,
без заяв про доведені польотні ймовірності. Групи сцен, профілів і режимів розділені.

Порівняння з visibility-v3 відображає спільну заміну satellite/DSM і SP/LightGlue
на aerial/DSM і RoMa. Воно не ізолює внесок карти або matcher. Спочатку
оцініть три рядки clear, далі — парні зміни за туману й диму.

`overview_by_condition.csv` містить підсумок усіх 24 кадрів для кожної умови.
`n_within_threshold` рахує всі точні PnP-позиції, включно з відхиленими;
`n_correct` — лише точні прийняті. `n_rejected_within_threshold` показує точні
позиції, які відхилив фільтр. `rejection_reasons.csv` деталізує причини відмов.
Шість умов одного кадру не є шістьма незалежними кадрами; висновки за сценами
потрібно перевіряти окремо. Це вибірка з орієнтаційним відбором, не весь датасет.


In [ ]:
from visibility_localization.analysis import summarize, summarize_batch, plot_results, diagnostic_figures
summary, paired = summarize(results, measured_output)
overview, rejection_reasons = summarize_batch(results, measured_output, CONFIG['success_threshold_m'])
overview['condition'] = pd.Categorical(overview.condition, categories=CONDITIONS, ordered=True)
print('Усі сцени: спочатку clear, потім погіршена видимість. Частки рахуються від усіх спроб.')
display(overview.sort_values('condition')[[
    'condition','n_frames','n_attempted','n_technical','n_pnp','n_within_threshold','n_correct',
    'n_false_accept','n_rejected','n_rejected_within_threshold','correct_fraction_all',
    'median_accepted_error_m','p95_accepted_error_m']])
print('За сценами: коректні прийняті позиції, хибні прийняття та відмови.')
display(summary[['scene_id','condition','n_attempted','n_correct','n_false_accept','n_rejected',
                 'n_technical','error_median_m']])
display(rejection_reasons)
paths = plot_results(results, measured_output)
from IPython.display import Image, display
for path in paths[:4]: display(Image(filename=str(path)))
if RUN_VISUAL_DIAGNOSTICS:
    reference = load_reference(diagnostic_selection[0]['reference_json'], CONFIG['reference_mode'])
    diagnostic_figures(backend, example, reference, CONFIG, output_root/'figures/examples')
    for path in sorted((output_root/'figures/examples').glob('*.png')):
        display(Image(filename=str(path)))


## 15. Обмеження та наступні етапи

- Реальні вимірювання з'являються лише після успішного виконання розділу 12.
  Синтетичні тести не перевіряють якість CAMP/RoMa на AnyVisLoc.
- Підтримка конкретного Colab, офіційних порталів і checkpoints перевіряється запуском.
  Помилка доступу не замінюється сторонніми даними або вигаданими показниками.
- Конвенція пози перевіряється на узгодженість матриці та Euler. Фізичне вирівнювання
  з реальною картою потребує перегляду діагностичних прикладів; невідомий випуск блокується.
- [Автори](https://github.com/UAV-AVL/Benchmark/blob/main/docs/GROUND_TRUTH.md)
  зазначають крок супутникового DSM 30 м, залишкові похибки прив'язки та нахил будівель.
  Локальні висоти не призначені для оцінювання абсолютної вертикальної локалізації.
- Плоска проєкція для географічної оцінки retrieval та розміру ділянок наближена;
  PnP використовує доступні висоти окремих точок. Чорні ділянки фільтруються авторською
  евристикою; вона не є достовірною маскою відсутніх даних.
- Радіус 3σXY не включає поширення кутової та висотної невизначеності.
  Профілі похибок не відповідають верифікованому датчику, INS або VIO.
- Географічний пошук обмежений заданою картою сцени, можливі виходи за карту,
  хибні кандидати, відмови та хибні прийняття; вони не виправляються еталоном.
- Порогові правила потребують окремої калібрувальної вибірки у наступному інкременті.
  На тестових умовах видимості пороги не переналаштовуються.
- Повторюваність залежить від версій і GPU. Недетермінована CUDA-операція спричиняє
  явну технічну помилку. Час залежить від кешу; ознака cache_hit зберігається.

Після технічної діагностики: перевірити реальні геометричні приклади, виконати обмежений
пакет, переглянути помилки за сценами, зафіксувати обсяг даних і невиконані критерії.
Навчання прогнозної моделі й планувальник залишаються наступними інкрементами.

- Yaw надходить із синтетичної попередньої орієнтації. Цей запуск не вимірює
  точність реального магнітометра/VIO. Кадри, prior, туман/дим і пороги
  збережено для порівняння з v5; поворот також змінює сітку кандидатів.


## 16. Порівняння з попереднім запуском без повороту карти
Ті самі 24 кадри, шість умов, prior і шум; окремо показано кожну спробу.


In [ ]:
# Порівняння з попередніми 144 спробами без повороту карти.
baseline_csv = BASELINE_ROOT/'batch/per_sample_results.csv'
if not baseline_csv.is_file():
    raise FileNotFoundError(f'Немає попередніх результатів: {baseline_csv}')
baseline = pd.read_csv(baseline_csv)
keys = ['scene_id','sample_id','condition','profile','sigma_xy_m','trial_id','retrieval_mode']
if len(baseline) != 144 or len(results) != 144:
    raise RuntimeError('Для порівняння потрібні два повні пакети по 144 спроби')
comparison = baseline.merge(results, on=keys, suffixes=('_before','_yaw'),
                            how='outer', validate='one_to_one', indicator=True)
if not comparison['_merge'].eq('both').all():
    raise RuntimeError('Набори спроб не збігаються')
for field in ['pair_id','prior_seed','smoke_seed','inference_seed']:
    if not comparison[field+'_before'].eq(comparison[field+'_yaw']).all():
        raise RuntimeError(f'Змінилися парні вимірювання: {field}')
comparison['correct_before'] = comparison.status_before.eq('correct').astype(int)
comparison['correct_yaw'] = comparison.status_yaw.eq('correct').astype(int)
comparison['delta_correct'] = comparison.correct_yaw-comparison.correct_before
comparison['condition'] = pd.Categorical(comparison.condition, categories=CONDITIONS, ordered=True)
by_condition = comparison.groupby('condition', observed=True).agg(
    attempts=('delta_correct','size'), correct_before=('correct_before','sum'),
    correct_yaw=('correct_yaw','sum'), delta_correct=('delta_correct','sum'))
print('Без повороту → поворот за noisy yaw; кількість коректних прийнятих позицій:')
display(by_condition)
print('Переходи статусів усіх спроб:')
display(pd.crosstab(comparison.status_before, comparison.status_yaw))
print('Кожна спроба: похибка, відповідності, inliers і причина відмови до/після:')
columns = keys[:3] + [name+suffix for name in
    ['status','error_xy_m','matches','inliers','inlier_ratio','rejection_reason']
    for suffix in ['_before','_yaw']]
display(comparison.sort_values(['condition','scene_id','sample_id'])[columns])
comparison.to_csv(measured_output/'paired_yaw_comparison.csv', index=False)
by_condition.to_csv(measured_output/'yaw_comparison_by_condition.csv')
